# Tracking: how well did one session track, and who corrected what

One night (or a window of it) from the driver log's PECLOG lines, plus the PHD2 guide log when there is one. Each
section answers one question:

1. **Where did the mount point?** Az / Alt / Roll over the session, and where each motor turned (MCU angle).
2. **Did the guide residuals shrink?** Sync-guide and pulse-guide residuals and their trend.
3. **Who corrected the drift?** The mount's drift split into pulse guiding, sync guiding, PEC drift (EMA) and the
   worm feed-forward, as running totals and rates.
4. **What did the guider see?** PHD2 RMS and effort per time block, PEC on vs off.
5. **Was PEC drift's time constant right for this night?** The driver's EMA replayed on the drift with other taus.
6. **How closely did the EMA follow the drift?** Its fit quality through the session.
7. **Did the worm feed-forward do its job?** The correction rebuilt from the motor angles, and the worm terms left
   for the guider.
8. **Optional: frame shift per motor** (KFLOG, needs `log_position = true`).
9. **Did M2 slip?** M2 creeping ahead and slipping back while it tracks (heavy payloads), and the guiding kicks it
   caused.

**Terms.** *Mount drift* is what the guide corrections and PEC added up to: the pointing error the mount would have
built without them (driver correction convention, arcsec on the RA and Dec axes). *PEC drift* is the driver's EMA of
the drift rate (`driver/control_pec.PecAxis`, `pec_tau_sec`), applied while `advanced_pec` is on. The *worm
feed-forward* (wff) corrects each motor's worm gear error from its MCU angle, from `data/worm_profile.json` (see
`analyse_worm_profile.ipynb`); the guide corrections lack what it corrected, so mount drift adds it back.

Cross-night questions (which tau suits most nights, does the worm repeat between nights) are in
`analyse_sessions.ipynb`.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
sys.path.insert(0, os.path.abspath(os.path.join('.', '..', 'driver')))
import importlib
import analyse_helpers, pe_analysis, worm_analysis, control_worm   # worm_analysis puts driver/ on the path
for m in (control_worm, analyse_helpers, pe_analysis, worm_analysis):   # dependencies first
    importlib.reload(m)
from analyse_helpers import (resolve_log_files, load_pec, load_kf_pid, load_phd2_guidelog, window_df,
                             load_sync_guiding_residuals, frame_shift_components, ramp_vs_next_step, correction_breakdown,
                             m2_slips, m2_slip_summary)
from catalog_logs import catalog_session
from pe_analysis import replay_pec_rate, rate_scores, true_rate, PecModel
from worm_analysis import wff_coupling, predicted_wff, worm_terms
from extract_segments import stitch_accum, DEFAULT_SITE
from control_pec import PecAxis
from sessions import get_session, list_sessions, site_latitude

TEMPLATE = 'plotly_dark'
rms = lambda s: float(np.sqrt(np.mean(np.square(s)))) if len(s) else np.nan

In [ ]:
# ── Choose the session ──────────────────────────────────────────────────────────────────────────
# Every captured log is registered once in utility/sessions.toml: its files, PHD2 guide log, default window and notes.
# New logs: copy them into logs/archive, run `uv run python sessions.py --add-new` in utility/, then edit the notes.
# What's available:  list_sessions()  /  list_sessions(records='pec')  /  list_sessions(contains='worm')
SESSION = 'jdm_Beta8.0_10_08_C62_NGC1365_300mm_worm'
S = get_session(SESSION)
log_filenames, LOG_DIR = S.log_filenames, S.log_dir
PHD2_GUIDELOG = S.phd2              # None skips section 4

# Optional time window (local time, as in the log); defaults to the session's t_from / t_to in sessions.toml
T_FROM, T_TO = S.t_from, S.t_to
# Optional: one automatically-detected tracking segment (listed below); overrides T_FROM / T_TO
SEGMENT = None
# Section 7: from when the worm profile in use stayed the same -- after the last driver restart or profile change
# (None = the window start). This session: the profile
# was re-applied at 21:34 after the last calibration run.
WORM_FROM = None # '2026-10-04 21:34'
SITE_LAT = site_latitude(SESSION, default=DEFAULT_SITE[0])

print(f"{SESSION}: {S.notes}\n  files {S.log_filenames}  PHD2 {'yes' if PHD2_GUIDELOG else 'no'}  "
      f"window {T_FROM}..{T_TO}  site latitude {SITE_LAT:.3f}")

In [ ]:
# ── Read PECLOG ─────────────────────────────────────────────────────────────────────────────────
resolved_files = resolve_log_files(log_filenames, log_dir=LOG_DIR)
for p in resolved_files:
    print(f"  {p}  ({os.path.getsize(p) / 1e6:.1f} MB)")
df, pec_config = load_pec(log_filenames, log_dir=LOG_DIR)

SEGMENTS = pd.DataFrame(catalog_session(resolved_files))
if len(SEGMENTS):
    print("Tracking segments (set SEGMENT = index to analyse one; under 1 min hidden):")
    seg_view = SEGMENTS[SEGMENTS['duration_min'] >= 1][['start', 'end', 'duration_min', 'start_reason', 'end_reason', 'pec',
                                                        'n_sync_guide', 'pulse_guiding', 'alt_mean', 'roll_mean']].copy()
    seg_view['start'] = seg_view['start'].dt.strftime('%m-%d %H:%M')
    seg_view['end'] = seg_view['end'].dt.strftime('%m-%d %H:%M')
    display(seg_view.round(1))
if SEGMENT is not None:
    T_FROM, T_TO = SEGMENTS.loc[SEGMENT, 'start'], SEGMENTS.loc[SEGMENT, 'end']
df = window_df(df, T_FROM, T_TO)
if df.empty:
    raise ValueError(f"No PECLOG rows inside the window T_FROM={T_FROM!r} T_TO={T_TO!r}")
if pec_config is None:                     # logs before PECCONFIG: the driver default
    pec_config = dict(mode='ema', tau=450.0, min_dt=0.05)
    print("No PECCONFIG line -- assuming the default EMA, tau 450 s")
if pec_config['mode'] != 'ema':
    print(f"This log ran PEC mode {pec_config['mode']!r} (since removed); sections 5-6 replay the EMA instead.")
print(f"PEC config {pec_config};  {df['timestamp'].iloc[0]} .. {df['timestamp'].iloc[-1]}  "
      f"({df['t_sec'].iloc[-1] / 3600:.2f} h, {len(df)} PECLOG rows)")

## 1. Where did the mount point?

Azimuth is shown around its median so a 0/360 crossing doesn't stretch the axis; real jumps (gotos) are left as gaps.
Roll matters for the worm feed-forward: near Roll 0 the M1 and M3 worms move the sky the same way.
Below it, each motor's MCU angle on its own scale, with how many worm turns (6°) it made in the window.

In [ ]:
GAP_S = 30           # PECLOG pauses longer than this (not tracking: slews, gotos) break the lines in section 1

def wrap_to_median(y, ref):
    return (y - ref + 180) % 360 - 180 + ref

def with_breaks(t, Y, jump=180.0, gap_s=GAP_S):
    """Times and columns (n, k) with a NaN row wherever PECLOG pauses (> gap_s) or a column jumps more than `jump`
    (an az wrap, an unwind remap), so the dotted lines leave a gap there without dropping a point. Also returns
    the break mask between consecutive rows."""
    t = pd.to_datetime(pd.Series(t)).reset_index(drop=True)
    Y = np.asarray(Y, float).reshape(len(t), -1)
    brk = (t.diff().dt.total_seconds().to_numpy()[1:] > gap_s) | np.any(np.abs(np.diff(Y, axis=0)) > jump, axis=1)
    at = np.flatnonzero(brk) + 1
    order = np.argsort(np.r_[np.arange(len(t)), at - 0.5], kind='stable')
    tx = np.r_[t.to_numpy(), t.to_numpy()[at]][order]
    return tx, np.vstack([Y, np.full((len(at), Y.shape[1]), np.nan)])[order], brk

def dotted(x, y, c, name):
    return go.Scatter(x=x, y=y, mode='lines+markers', marker=dict(size=2, color=c), line=dict(color=c),
                      name=name, showlegend=False)

COLOURS = ('royalblue', 'orange', 'mediumseagreen')
pose = np.column_stack([wrap_to_median(df['az'], df['az'].median()), df['alt'], df['roll']])
tx, pose_b, _ = with_breaks(df['timestamp'], pose)
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['Azimuth (deg, around its median)', 'Altitude (deg)', 'Roll (deg)'])
for r, (name, c) in enumerate(zip(('Az', 'Alt', 'Roll'), COLOURS)):
    fig.add_trace(dotted(tx, pose_b[:, r], c, name), row=r + 1, col=1)
fig.update_layout(height=650, width=1200, template=TEMPLATE, title='Mount orientation over the session')
fig.show()


In [ ]:
# Where the motors moved: each motor's MCU angle zeta (theta_raw - zeta_offset, the worm feed-forward's angle), own
# scale per motor. Breaks as above (pauses, and jumps over 180 deg: an unwind remap); worm turns count only the motion
# within the unbroken stretches (not slews, gotos or remaps).
if all(f'theta_raw_{i}' in df.columns for i in (1, 2, 3)):
    mot = df.dropna(subset=['theta_raw_1', 'theta_raw_2', 'theta_raw_3']).reset_index(drop=True)
    ang = (worm_analysis.motor_angles(mot) if 'zeta_offset_1' in mot.columns
           else mot[['theta_raw_1', 'theta_raw_2', 'theta_raw_3']].to_numpy(float))
    tx, ang_b, brk = with_breaks(mot['timestamp'], ang)
    turns = np.abs(np.diff(ang, axis=0)[~brk]).sum(axis=0) / worm_analysis.WORM_THETA
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                        subplot_titles=[f'M{i + 1} zeta (deg): {turns[i]:.1f} worm turns' for i in range(3)])
    for i, c in enumerate(COLOURS):
        fig.add_trace(dotted(tx, ang_b[:, i], c, f'M{i + 1}'), row=i + 1, col=1)
    fig.update_layout(height=650, width=1200, template=TEMPLATE, title='Motor angles over the session (MCU angle zeta)')
    fig.show()
else:
    print("No theta_raw in this log -- motor angles skipped.")


## 2. Did the guide residuals shrink?

Each guide correction's residual (what the guider still had to correct). A sync-guide row carries both axes, a
pulse-guide row one. With PEC drift learning the drift, the rolling RMS should come down after the first ~15 min.

In [ ]:
def axis_resid(col):
    r = df.dropna(subset=[col]).sort_values('t_sec').reset_index(drop=True)
    r['arcsec'] = r[col] * 60
    n = max(5, len(r) // 20)
    r['rms_roll'] = r['arcsec'].rolling(n, min_periods=1).apply(lambda x: np.sqrt(np.mean(x ** 2)))
    return r, n

def trend_summary(r, label):
    t, y = r['t_sec'].values, r['arcsec'].values
    if len(t) < 4:
        return dict(axis=label, corrections=len(t))
    k = max(1, len(y) // 5)
    first, last = rms(y[:k]), rms(y[-k:])
    return {'axis': label, 'corrections': len(y), 'trend "/h': np.polyfit(t, y, 1)[0] * 3600,
            'first 20% rms "': first, 'last 20% rms "': last, 'change %': 100 * (last - first) / first if first else np.nan}

resid = {ax: axis_resid(col) for ax, col in (('RA', 'resid_1'), ('Dec', 'resid_2'))}
display(pd.DataFrame([trend_summary(r, ax) for ax, (r, _) in resid.items()]).round(2))
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                    subplot_titles=[f'{ax} guide residual (arcsec)' for ax in resid])
for i, (ax, (r, n)) in enumerate(resid.items(), start=1):
    fig.add_trace(go.Scatter(x=r['timestamp'], y=r['arcsec'], mode='markers', name=f'{ax} residual',
                             marker=dict(size=4, opacity=0.4, color='royalblue' if i == 1 else 'mediumseagreen')), row=i, col=1)
    fig.add_trace(go.Scatter(x=r['timestamp'], y=r['rms_roll'], name=f'{ax} rolling RMS (n={n})',
                             line=dict(color='orange', width=2)), row=i, col=1)
    fig.add_hline(y=0, line=dict(color='gray', width=1, dash='dot'), row=i, col=1)
fig.update_layout(height=650, width=1200, template=TEMPLATE, hovermode='x unified', title='Guide residuals')
fig.show()

## 3. Who corrected the drift?

Running totals per axis: **mount drift = pulse guiding + sync guiding + PEC drift applied + worm feed-forward**
(joined across PEC resets), then each as a rate (slope over ±5 min). PEC is doing its job when its rate follows the
drift rate and the guiders' rates stay near zero; when the guider's total runs opposite to PEC's, they are fighting.
Shaded in the bottom panel: PEC inhibited (not applying).

In [ ]:
pvg = correction_breakdown(df, rate_half_window_s=300)
COL = {'mount drift': 'white', 'pulse guiding': 'royalblue', 'sync guiding': 'orange', 'PEC drift': 'magenta',
       'worm FF': 'lime'}
PARTS = [('mount_drift', 'mount drift'), ('pulse_guide', 'pulse guiding'), ('sync_guide', 'sync guiding'),
         ('pec_applied', 'PEC drift'), ('worm_ff', 'worm FF')]
summary = []
for ax, name in (('ra', 'RA'), ('dec', 'Dec')):
    x = pvg[ax]
    if x.empty:
        print(f"{name}: no guide corrections")
        continue
    tx = x['timestamp']
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
        subplot_titles=[f'{name} running totals (arcsec)', f'{name} rates (arcsec/min, ±5 min slopes)',
                        f'{name} PEC drift: its model rate and what it applied vs the actual drift rate (arcsec/min)'])
    for col, label in PARTS:
        if col == 'worm_ff' and not np.any(x[col] != 0):
            continue
        w = 3 if col == 'mount_drift' else 1.5
        fig.add_trace(go.Scatter(x=tx, y=x[col], name=f'{name} {label}', legendgroup=label,
                                 line=dict(color=COL[label], width=w)), row=1, col=1)
        rate = 'drift_rate' if col == 'mount_drift' else f'{col}_rate'
        fig.add_trace(go.Scatter(x=tx, y=x[rate], name=f'{name} {label} rate', legendgroup=label, showlegend=False,
                                 line=dict(color=COL[label], width=w)), row=2, col=1)
    fig.add_trace(go.Scatter(x=tx, y=x['drift_rate'], showlegend=False, line=dict(color='white', width=3)), row=3, col=1)
    fig.add_trace(go.Scatter(x=tx, y=x['pec_model_rate'], name=f'{name} PEC model rate', line=dict(color='red')), row=3, col=1)
    fig.add_trace(go.Scatter(x=tx, y=np.where(x['pec_active'], x['pec_applied_logged_rate'], 0.0),
                             name=f'{name} PEC applied rate', line=dict(color='magenta', width=1.5)), row=3, col=1)
    inactive = ~x['pec_active'].values.astype(bool)
    edges = np.flatnonzero(np.diff(np.r_[0, inactive.astype(int), 0]))
    for a, b in zip(edges[::2], edges[1::2]):
        fig.add_vrect(x0=tx.iloc[a], x1=tx.iloc[min(b, len(tx) - 1)], fillcolor='gray', opacity=0.15, line_width=0, row=3, col=1)
    for r in (2, 3):
        fig.add_hline(y=0, line=dict(color='gray', width=1), row=r, col=1)
    for tr in pvg['resets']:
        fig.add_vline(x=tr, line=dict(color='gray', width=1, dash='dot'))
    fig.update_layout(height=950, width=1400, template=TEMPLATE, hovermode='x unified',
                      title=f'{name}: who corrected the drift (dotted: PEC resets)')
    fig.show()

    last = x.iloc[-1]
    guider, pec = last['pulse_guide'] + last['sync_guide'], last['pec_applied']
    m = ~np.isnan(x['drift_rate'].values)
    summary.append({'axis': name, 'mount drift "': last['mount_drift'], 'pulse guiding "': last['pulse_guide'],
                    'sync guiding "': last['sync_guide'], 'PEC drift "': pec, 'worm FF "': last['worm_ff'],
                    'PEC share %': pec / last['mount_drift'] * 100 if last['mount_drift'] else np.nan,
                    'PEC undone by guider %': -guider / pec * 100 if pec and np.sign(guider) != np.sign(pec) else 0,
                    'PEC active %': x['pec_active'].mean() * 100,
                    'model rate error rms "/min': rms((x['pec_model_rate'] - x['drift_rate'])[m]),
                    'drift rate rms "/min': rms(x['drift_rate'][m])})
summary = pd.DataFrame(summary)
display(summary.round(1))
for _, r in summary.iterrows():
    if r['PEC share %'] < 0:
        print(f"{r['axis']}: PEC pushed against the drift")
    if r['PEC undone by guider %'] > 50:
        print(f"{r['axis']}: the guider undid {r['PEC undone by guider %']:.0f}% of what PEC applied -- they are fighting")
    if r['model rate error rms "/min'] > r['drift rate rms "/min']:
        print(f"{r['axis']}: PEC's rate was further from the drift rate than zero would have been")

## 4. What did the guider see?

PHD2's own measure of the result: RMS per time block and the guider's effort (pulse ms per minute), with PEC on vs
off. Frames just after a sync are listed separately: a sync moves the target, so PHD2 sees a step it has to guide out.

In [ ]:
BLOCK = '20min'
SYNC_SETTLE = pd.Timedelta('3min')

if not PHD2_GUIDELOG:
    print("No PHD2 guide log -- skipped.")
else:
    frames, events = load_phd2_guidelog(PHD2_GUIDELOG)
    frames, events = window_df(frames, T_FROM, T_TO), window_df(events, T_FROM, T_TO)
    if frames.empty:
        print("No PHD2 guided frames inside the window")
    else:
        fr = frames.copy()
        active = df[df['pec_active'].astype(bool)] if 'pec_active' in df.columns else df
        on = pd.merge_asof(fr[['timestamp']].astype('datetime64[ns]'),
                           active[['timestamp']].astype('datetime64[ns]').assign(pec_on=True),
                           on='timestamp', direction='nearest', tolerance=pd.Timedelta('60s'))
        fr['pec_on'] = on['pec_on'].astype('boolean').fillna(False).astype(bool).values
        try:
            sync_times = window_df(load_sync_guiding_residuals(log_filenames, log_dir=LOG_DIR), T_FROM, T_TO)['timestamp']
        except ValueError:
            sync_times = pd.Series([], dtype='datetime64[ns]')
        fr['after_sync'] = False
        for t in sync_times:
            fr.loc[(fr['timestamp'] >= t) & (fr['timestamp'] < t + SYNC_SETTLE), 'after_sync'] = True

        fr['block'] = fr['timestamp'].dt.floor(BLOCK)
        g = fr.groupby('block')
        minutes = g['timestamp'].agg(lambda s: max((s.iloc[-1] - s.iloc[0]).total_seconds() / 60, 1e-9))
        lost = events[events['kind'] == 'star_lost'].groupby(events['timestamp'].dt.floor(BLOCK)).size() \
            if not events.empty else pd.Series(dtype=int)
        blocks = pd.DataFrame({
            'PEC on %': (g['pec_on'].mean() * 100).round(0),
            'RA rms "': g['ra_arcsec'].agg(rms).round(2),
            'Dec rms "': g['dec_arcsec'].agg(rms).round(2),
            'total rms "': g.apply(lambda x: rms(np.hypot(x['ra_arcsec'], x['dec_arcsec'])), include_groups=False).round(2),
            'RA pulses ms/min': (g['ra_ms'].agg(lambda s: s.abs().sum()) / minutes).round(0),
            'Dec pulses ms/min': (g['dec_ms'].agg(lambda s: s.abs().sum()) / minutes).round(0),
            'syncs': sync_times.groupby(sync_times.dt.floor(BLOCK)).size(),
            'star lost': lost,
        }).loc[minutes.index].fillna({'syncs': 0, 'star lost': 0}).astype({'syncs': int, 'star lost': int})
        display(blocks)

        rows = []
        for label, sel in (('PEC off', ~fr['pec_on']), ('PEC on', fr['pec_on'])):
            for excl, mask in (('all frames', sel), (f'not within {SYNC_SETTLE.total_seconds() / 60:.0f} min of a sync',
                                                     sel & ~fr['after_sync'])):
                x = fr[mask]
                if len(x):
                    rows.append({'PEC': label, 'frames': excl, 'n': len(x), 'RA rms "': rms(x['ra_arcsec']),
                                 'Dec rms "': rms(x['dec_arcsec']), 'total rms "': rms(np.hypot(x['ra_arcsec'], x['dec_arcsec']))})
        display(pd.DataFrame(rows).round(2))

        fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                            subplot_titles=[f'Guiding RMS per {BLOCK} (arcsec)', 'Guider effort (pulse ms per minute)'])
        for col, c in (('RA', 'royalblue'), ('Dec', 'mediumseagreen')):
            fig.add_trace(go.Bar(x=blocks.index, y=blocks[f'{col} rms "'], name=f'{col} rms', marker_color=c), row=1, col=1)
            fig.add_trace(go.Bar(x=blocks.index, y=blocks[f'{col} pulses ms/min'], name=f'{col} pulses', marker_color=c,
                                 showlegend=False), row=2, col=1)
        fig.add_trace(go.Scatter(x=blocks.index, y=blocks['total rms "'], name='total rms', mode='lines+markers',
                                 line=dict(color='white')), row=1, col=1)
        for t in sync_times:
            fig.add_vline(x=t, line=dict(color='orange', width=1, dash='dot'))
        for b in blocks.index[blocks['PEC on %'] >= 50]:
            fig.add_vrect(x0=b, x1=b + pd.Timedelta(BLOCK), fillcolor='magenta', opacity=0.08, line_width=0)
        fig.update_layout(height=700, width=1400, template=TEMPLATE, barmode='group', hovermode='x unified',
                          title='Guider outcome (shaded: PEC on, dotted: syncs)')
        fig.show()

## 5. Was PEC drift's time constant right for this night?

The driver's EMA (`PecAxis`) replayed causally on this night's mount drift with a range of time constants. Scored by
the drift left for the guider over a guide interval (the step in drift minus rate x interval), relative to no PEC:
below 1 helps, above 1 hurts. A short tau follows changes quickly but passes noise; a long one is smooth but lags.
The driver default is 450 s (7.5 min); `analyse_sessions.ipynb` makes the same comparison across all nights.

In [ ]:
BENCH_INTERVALS = (30, 120, 300)          # s
THIN_S = 5.0                              # pulse-guided PECLOG is ~1 Hz; PecAxis uses each sample's own dt
TAUS_MIN = (1, 2, 5, 7.5, 10, 15, 21, 30, 60)
d = df.dropna(subset=['total_accum_1', 'total_accum_2']).sort_values('t_sec')
reset = np.r_[False, np.diff(d['n'].values) < 0]
t_all = d['t_sec'].values
drift = {'ra': stitch_accum(d['total_accum_1'].values * 60, reset), 'dec': stitch_accum(d['total_accum_2'].values * 60, reset)}
keep = np.r_[True, np.diff(np.floor(t_all / THIN_S)) > 0]
t_b = t_all[keep] - t_all[0]

session_model = PecModel(f"this session (tau {pec_config['tau'] / 60:g}m)", float(pec_config['tau']))
BENCH_MODELS = [session_model] + [PecModel(f'tau {m:g}m', m * 60) for m in TAUS_MIN]
bench_rates, rows = {}, []
for ax in ('ra', 'dec'):
    y = drift[ax][keep]
    rates = {m.name: replay_pec_rate(t_b, y, m) for m in BENCH_MODELS}
    bench_rates[ax] = (y, rates)
    for dt, res in rate_scores(t_b, y, rates, guide_intervals_s=BENCH_INTERVALS)['guide_rms'].items():
        rows += [dict(axis=ax, interval=f'{dt} s', model=name, rms=v) for name, v in res.items()]
bench = pd.DataFrame(rows)
pooled = bench.pivot_table(index='model', columns='interval', values='rms', aggfunc=lambda v: np.sqrt(np.mean(np.square(v))))
rel = pooled.div(pooled.loc['no_pec'], axis=1).reindex(['no_pec'] + [m.name for m in BENCH_MODELS])[[f'{dt} s' for dt in BENCH_INTERVALS]]
print(f"{t_b[-1] / 3600:.1f} h of drift, {len(t_b)} samples. Drift left for the guider per interval, relative to no PEC:")
display(rel.round(3))
best = rel.loc[[m.name for m in BENCH_MODELS[1:]], '120 s'].idxmin()
print(f"best for a 2-min interval: {best} ({rel.loc[best, '120 s']:.2f}); this session's: {rel.loc[session_model.name, '120 s']:.2f}")

AXIS = 'ra'
y, rates = bench_rates[AXIS]
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_b / 60, y=true_rate(t_b, y, 300), name='actual drift rate (±5 min)', line=dict(color='white', width=3)))
for name, c in zip([session_model.name, 'tau 2m', 'tau 21m'], ('red', 'mediumseagreen', 'royalblue')):
    fig.add_trace(go.Scatter(x=t_b / 60, y=rates[name], name=name, line=dict(color=c)))
fig.update_layout(height=420, width=1400, template=TEMPLATE, hovermode='x unified', xaxis_title='minutes',
                  yaxis_title='arcsec/min', title=f'{AXIS.upper()} drift rate: actual vs what each tau would have applied')
fig.show()

## 6. How closely did the EMA follow the drift?

The session's EMA replayed row by row (restarting at each PEC reset, as the driver does). R² is the driver's own fit
quality, the one its inhibit gate uses: PEC applies only while R² and the rmse are good enough. *Drift minus model* is
what the EMA's prediction missed at each guide update.

In [ ]:
VAR_ALPHA, SSE_ALPHA = 0.05, 0.15
axes = [PecAxis(tau=pec_config['tau'], min_dt=pec_config['min_dt']) for _ in range(2)]
ra_drift, dec_drift = (stitch_accum(df[f'total_accum_{i}'].values * 60, np.r_[False, np.diff(df['n'].values) < 0]) for i in (1, 2))
offset = [ra_drift - df['total_accum_1'].values * 60, dec_drift - df['total_accum_2'].values * 60]
for k, a in enumerate(axes):
    a.reset_seed(df.iloc[0][f'total_accum_{k + 1}'] / 60)
t0, prev_n, rec = 0.0, df.iloc[0]['n'], []
for i, row in enumerate(df.itertuples()):
    if row.n < prev_n:                                          # PEC reset: the driver restarts the model and its clock
        for k, a in enumerate(axes):
            a.reset()
            a.reset_seed(getattr(row, f'total_accum_{k + 1}') / 60)
        t0 = row.t_sec
    prev_n = row.n
    t = row.t_sec - t0
    r = dict(timestamp=row.timestamp)
    for k, (a, nm) in enumerate(zip(axes, ('ra', 'dec'))):
        accum = getattr(row, f'total_accum_{k + 1}') / 60
        if np.isfinite(accum):
            a.ingest_accum(accum, t, VAR_ALPHA, SSE_ALPHA)
        r[f'{nm}_r2'] = a.r2
        r[f'{nm}_miss'] = (accum - a.predicted_accum(t)) * 3600
        r[f'{nm}_rmse'] = np.sqrt(a.sse) * 3600
    rec.append(r)
fit = pd.DataFrame(rec)
fig = make_subplots(rows=2, cols=2, shared_xaxes=True, vertical_spacing=0.12,
                    subplot_titles=['RA R²', 'Dec R²', 'RA drift minus model, and rmse (arcsec)', 'Dec drift minus model, and rmse (arcsec)'])
for c, nm in ((1, 'ra'), (2, 'dec')):
    fig.add_trace(go.Scatter(x=fit['timestamp'], y=fit[f'{nm}_r2'], name=f'{nm} R²', line=dict(color='royalblue')), row=1, col=c)
    fig.add_trace(go.Scatter(x=fit['timestamp'], y=fit[f'{nm}_miss'], name=f'{nm} drift minus model', line=dict(color='royalblue')), row=2, col=c)
    fig.add_trace(go.Scatter(x=fit['timestamp'], y=fit[f'{nm}_rmse'], name=f'{nm} rmse', line=dict(color='red', dash='dash')), row=2, col=c)
    fig.add_hrect(y0=0, y1=0.5, fillcolor='rgba(255,0,0,0.15)', line_width=0, row=1, col=c)
    fig.update_yaxes(range=[0, 1], row=1, col=c)
fig.update_xaxes(matches='x')
fig.update_layout(height=600, width=1400, template=TEMPLATE, hovermode='x unified', showlegend=False,
                  title=f"PEC drift EMA, tau {pec_config['tau']:g} s (red band: R² below 0.5)")
fig.show()

## 7. Did the worm feed-forward do its job?

The worm feed-forward turns each motor's worm profile (a sine on its MCU angle, period 6°) into a pointing correction
and logs it per row as `wff` (RA and Dec arcmin). Three checks, from the window where the profile in use stayed the
same (`WORM_FROM`):

* **Is the logged correction the profile?** Rebuild it from the motor angles and the pose; fitting the worm terms to
  the logged wff returns the profile that was in use.
* **What did the profile leave for the guider?** The same fit on the guide corrections (`total_accum + pec_accum`,
  joined across PEC resets, with a smooth drift and an offset per PEC piece): any worm term here is worm error the
  profile did not remove. Ideally all amplitudes are within ~2 SE of zero.
* **What did the mount need?** Guide corrections plus the feed-forward: the worm the profile should have had. Compare
  it with the calibration's result in `analyse_worm_profile.ipynb`.

A motor that hardly turned (a few worm periods or less) can't be separated from the drift: check *turns*. The
amplitudes are arcsec of motor angle, the profile's units; the SE from a night of tracking is optimistic too (the
residuals are correlated in time), so treat differences under ~3 SE with care.

In [ ]:
w = window_df(df, WORM_FROM, None) if WORM_FROM else df
w = w.dropna(subset=['theta_raw_1', 'wff_1', 'total_accum_1', 'total_accum_2']).reset_index(drop=True)
if w.empty or not np.any(w[['wff_1', 'wff_2']].abs().to_numpy() > 0):
    print("No worm feed-forward in this window (no profile in use, or a log from before it) -- skipped.")
else:
    C, align_err = wff_coupling(w, SITE_LAT)
    turns = [round(float(x), 1) for x in np.ptp(worm_analysis.motor_angles(w), axis=0) / worm_analysis.WORM_THETA]
    print(f"{len(w)} PECLOG rows from {w['timestamp'].iloc[0]:%H:%M} to {w['timestamp'].iloc[-1]:%H:%M}; "
          f"alignment from the pose fits to {align_err:.1f}' (median); worm turns M1/M2/M3 {turns}")
    reset = np.r_[False, np.diff(w['n'].values) < 0]
    wff = w[['wff_1', 'wff_2']].to_numpy(float)
    guide = np.column_stack([w[f'total_accum_{i}'].to_numpy(float) + w[f'pec_accum_{i}'].fillna(0).to_numpy(float)
                             for i in (1, 2)])
    series = {'logged wff (the profile in use)': wff, 'left for the guider': guide, 'needed (guide + wff)': guide + wff}
    rows = []
    for label, y in series.items():
        terms, r = worm_terms(w, y, C)
        for _, t in terms.iterrows():
            rows.append({'series': label, 'motor': t['motor'], 'harmonic': t['harmonic'], 'a': t['a'], 'b': t['b'],
                         'amplitude "': t['amplitude'], 'phase deg': t['phase'], 'peak deg': t['peak'], 'se "': t['se'],
                         'amp / se': t['amplitude'] / t['se'], 'reliable': t['reliable'], "fit rms '": r})
    wt = pd.DataFrame(rows)
    wt['amplitude ± se @ peak'] = [f'{a:5.1f}" ± {e:.1f} @ {p:5.1f}°' + ('' if ok else ' ⚠')
                                    for a, e, p, ok in zip(wt['amplitude "'], wt['se "'], wt['peak deg'], wt['reliable'])]
    display(wt.pivot(index=['motor', 'harmonic'], columns='series', values='amplitude ± se @ peak')[list(series)])
    sep = worm_analysis.separation_deg(C)
    print(f"M1/M3 separation {np.percentile(sep, 10):.0f}-{sep.max():.0f} deg (10th percentile to max); "
          "⚠ = fewer than 2 worm turns, or M1/M3 too close to tell apart: don't trust that term")
    print("fit rms per series ('):", {k: round(v, 2) for k, v in wt.groupby('series', sort=False)["fit rms '"].first().items()})
    for M, t in zip(('M1', 'M2', 'M3'), turns):
        if t < 3:
            print(f"{M} turned only {t:.1f} worm periods: its terms are hard to tell from the drift")

    # the logged wff against the one rebuilt from the profile that the fit returned
    logged = wt[wt['series'].str.startswith('logged')]
    coef = np.array([[float(logged[(logged['motor'] == M) & (logged['harmonic'] == h)][k].iloc[0])
                      for h in (1, 2) for k in ('a', 'b')] for M in ('M1', 'M2', 'M3')])
    pred = predicted_wff(w, C, coef, harmonics=(1, 2))
    for k, ax in enumerate(('RA', 'Dec')):
        print(f"wff {ax}: logged rms {np.std(wff[:, k]) * 60:.1f}\"  rebuilt-minus-logged rms "
              f"{np.sqrt(np.mean((pred[:, k] - wff[:, k]) ** 2)) * 60:.2f}\"  corr {np.corrcoef(pred[:, k], wff[:, k])[0, 1]:.3f}")

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                        subplot_titles=['RA (arcsec)', 'Dec (arcsec)'])
    for k in range(2):
        fig.add_trace(go.Scatter(x=w['timestamp'], y=wff[:, k] * 60, name='logged wff', legendgroup='l', showlegend=k == 0,
                                 line=dict(color='lime')), row=k + 1, col=1)
        fig.add_trace(go.Scatter(x=w['timestamp'], y=pred[:, k] * 60, name='rebuilt from the motor angles', legendgroup='p',
                                 showlegend=k == 0, line=dict(color='white', dash='dot')), row=k + 1, col=1)
        g = stitch_accum(guide[:, k] * 60, reset)
        trend = np.polyval(np.polyfit(w['t_sec'], g, 3), w['t_sec'])
        fig.add_trace(go.Scatter(x=w['timestamp'], y=g - trend, name='guide corrections, drift removed', legendgroup='g',
                                 showlegend=k == 0, line=dict(color='orange', width=1)), row=k + 1, col=1)
    fig.update_layout(height=650, width=1400, template=TEMPLATE, hovermode='x unified',
                      title='Worm feed-forward and what the guider still corrected')
    fig.show()

    fig = go.Figure()
    for label, sym in zip(series, ('circle', 'x', 'diamond')):
        s = wt[(wt['series'] == label) & (wt['harmonic'] == 1)]
        a = s['amplitude "'] * np.cos(np.radians(s['phase deg']))
        b = s['amplitude "'] * np.sin(np.radians(s['phase deg']))
        fig.add_trace(go.Scatter(x=a, y=b, mode='markers+text', text=s['motor'], textposition='top center', name=label,
                                 marker=dict(symbol=sym, size=12), error_x=dict(array=s['se "']), error_y=dict(array=s['se "'])))
    fig.add_hline(y=0, line=dict(color='gray', width=1)); fig.add_vline(x=0, line=dict(color='gray', width=1))
    fig.update_layout(height=600, width=700, template=TEMPLATE, xaxis_title='a (sin), arcsec', yaxis_title='b (cos), arcsec',
                      yaxis_scaleanchor='x', title='Worm terms as phasors: the distance between two points is the worm left')
    fig.show()

## 8. Optional: frame shift per motor (KFLOG)

Needs `log_position = true`. The base-vs-corrected frame shift per motor splits into steps at syncs (orange) and the
change between syncs (cyan, what PEC and the feed-forward apply). When the next sync keeps undoing the change between
syncs, what is applied between them pushes against the drift.

In [ ]:
kf_df, _ = load_kf_pid(log_filenames, log_dir=LOG_DIR)
kf_df = window_df(kf_df, T_FROM, T_TO)
shift_cols = [f'θ_meas_raw_{i}' for i in (1, 2, 3)] + [f'θ_ref_raw_{i}' for i in (1, 2, 3)]
if not len(kf_df) or not all(c in kf_df.columns for c in shift_cols):
    print("No KFLOG θ_meas_raw / θ_ref_raw in this capture -- skipped (needs log_position = true).")
else:
    kf = kf_df.dropna(subset=shift_cols).sort_values('timestamp').reset_index(drop=True)
    try:
        kf_syncs = window_df(load_sync_guiding_residuals(log_filenames, log_dir=LOG_DIR), T_FROM, T_TO)['timestamp']
    except ValueError:
        kf_syncs = pd.Series([], dtype='datetime64[ns]')
    kf_syncs = kf_syncs[(kf_syncs >= kf['timestamp'].iloc[0]) & (kf_syncs <= kf['timestamp'].iloc[-1])]
    comp, shift_summary = frame_shift_components(kf, kf_syncs)
    print(f"{len(kf)} KFLOG samples, {len(kf_syncs)} syncs; median θ2 {shift_summary.attrs['theta2_deg']:.1f}° -> "
          f"M1/M3 amplification x{shift_summary.attrs['m1_m3_amplification']:.1f}")
    display(shift_summary.round(1))
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                        subplot_titles=[f'M{i}: shift (arcsec) = steps at syncs + change between syncs' for i in (1, 2, 3)])
    for i in (1, 2, 3):
        m = f'M{i}'
        fig.add_trace(go.Scatter(x=comp['timestamp'], y=comp[f'{m} shift'], name=f'{m} shift', line=dict(color='gray', width=1)), row=i, col=1)
        fig.add_trace(go.Scatter(x=comp['timestamp'], y=comp[f'{m} staircase'] + comp[f'{m} shift'].iloc[:50].mean(),
                                 name=f'{m} steps at syncs', line=dict(color='orange', width=2)), row=i, col=1)
        fig.add_trace(go.Scatter(x=comp['timestamp'], y=comp[f'{m} smooth'], name=f'{m} between syncs', line=dict(color='cyan', width=2)), row=i, col=1)
    fig.update_layout(height=850, width=1400, template=TEMPLATE, hovermode='x unified', title='Base vs corrected frame shift')
    fig.show()
    rvs = ramp_vs_next_step(comp, kf_syncs)
    display(rvs.round(2))
    for m, r in rvs.iterrows():
        if r['intervals'] >= 5 and r['corr'] < -0.5 and r['share undone'] > 0.5:
            print(f"{m}: the next sync undoes {r['share undone'] * 100:.0f}% of each ramp -- what's applied between syncs pushes against the drift")

## 9. Did M2 slip?

With a heavy payload, M2 can creep ahead and then slip back 10-30" at once while it tracks, and the guider sees each slip as a sharp kick in RA and Dec (8 Oct 2026, 300 mm f/2.8: every 10-12 s while M2 lifted the lens at 8-15"/s). Each slip shows in PECLOG's M2 angle as a step back against its motion (over `SLIP_ARCSEC`, 8" by default). The table counts them per hour of tracking by M2's speed and whether its motion lifts or lowers the lens; with a PHD2 guide log, also how many the guider saw as a jump. Many slips while lifting: choose the **ZM2** roll on the Dashboard (see the Guiding guide, section 6.3).

Needs PECLOG rows about 1 s apart (pulse guiding); sync-guided stretches are too sparse and are left out.

In [ ]:
steps = m2_slips(df)
if steps.empty:
    print("No PECLOG rows close enough together (pulse guiding, ~1 s) -- skipped.")
else:
    phd_frames = None
    if PHD2_GUIDELOG:
        phd_frames = window_df(load_phd2_guidelog(PHD2_GUIDELOG)[0], T_FROM, T_TO)
    slips = steps[steps['slip']]
    hours = steps['dt'].sum() / 3600
    print(f"{len(slips)} M2 slips in {hours:.1f} h of tracking ({len(slips) / hours:.0f} per hour), "
          f"median {slips['size'].median():.0f}\" back" if len(slips) else f"No M2 slips in {hours:.1f} h of tracking")
    display(m2_slip_summary(steps, phd_frames).round(1))
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                        subplot_titles=['M2 speed ("/s; + is M2 turning up)', 'M2 slips: size (arcsec back)'])
    fig.add_trace(go.Scatter(x=steps['timestamp'], y=steps['rate'], mode='lines', name='M2 speed',
                             line=dict(color='royalblue')), row=1, col=1)
    colour = {'lifting': 'orangered', 'lowering': 'mediumseagreen', 'still': 'gray'}
    for direction, g in slips.groupby('direction'):
        fig.add_trace(go.Scatter(x=g['timestamp'], y=g['size'], mode='markers', name=f'slip while {direction}',
                                 marker=dict(size=5, color=colour.get(direction, 'white'))), row=2, col=1)
    fig.update_layout(height=550, width=1200, template=TEMPLATE, title='M2 slips over the session')
    fig.show()